# TrueID Live Commerce Copilot

## 1. Setup

**Solution capability:** an AI live-commerce copilot for True Digital Group and True Corp that turns Thai live-stream speech into synchronized captions and commerce actions. The current MVP focuses on ASR, product/promotion grounding from transcript text, product cards, bundle recommendations, and flash-sale countdowns. Product visual grounding and viewer Q&A are intentionally out of scope for this slice.

**Strategic value for True:** the capability can sit inside TrueID live commerce, TrueVisions shopping formats, creator storefronts, and telco bundle campaigns. It helps hosts convert spoken selling moments into clickable product/promo UI, increases basket attach through bundles, and creates a reusable AI layer for commerce, advertising, and partner brand activations.

**Evaluation alignment:**

- **Creativity & Innovation:** combines Thai live ASR, transcript-driven commerce action extraction, promo eligibility validation, and synchronized shopping UI for live commerce.
- **Business Impact & ROI:** targets higher conversion, larger average order value through bundle prompts, faster campaign setup for sellers, and lower manual live-operation effort.
- **Code Quality & Engineering:** uses a structured repository with `src/`, `data/`, `config/`, `tests/`, reusable notebook UI helpers, deterministic cached sample outputs, and automated tests.
- **AI Technique & Scalability:** uses pause-aware streaming chunks, OpenAI Whisper turbo by default, deterministic action decisions, catalog/promotion validation, and a path to RAG/embedding retrieval for larger catalogs.

**Execution guardrail:** run this setup cell first. It finds or clones the GitHub repo, installs dependencies, adds the repo to `sys.path`, and preloads OpenAI Whisper turbo so the demo cells can run more reliably in Colab.


### 1.1 Current Demo Architecture

The MVP is organized as a small production-shaped system rather than a single script:

| Layer | Current module/data | Responsibility |
| --- | --- | --- |
| Notebook UI | `src/utils/demo_notebook_ui.py` | Renders catalog, recording-mode, live-file, and live-mic demo surfaces. Keeps notebook cells short and reusable. |
| ASR | `src/ai/captioning.py`, `src/utils/live_mic.py`, `src/utils/realtime_audio_file.py` | Converts Thai/English audio into timestamped caption segments. Supports cached samples and OpenAI Whisper. |
| Catalog & promos | `data/demo/catalog/product_catalog.csv`, `data/demo/catalog/promotions.csv`, `src/data/catalog.py` | Loads products and promotion rules. Promotions are separate from products so eligibility can be validated centrally. |
| Retrieval | `src/ai/retrieval.py` | Finds product and promo candidates from noisy transcript text with normalized Thai/English text, Thai-digit handling, character n-gram similarity, substring matching, and price/promo-number boosts. |
| Decision layer | `src/ai/decision.py` | Chooses structured commerce decisions from transcript history, candidates, and active session state using deterministic rules validated against catalog and promotion data. |
| Action engine | `src/ai/commerce_actions.py` | Validates decisions against catalog/promo rules and emits timestamped UI actions: product pin, promo code, bundle, and flash-sale countdown. |
| Outputs | `outputs/...` | Writes captions/actions JSON and timeline artifacts for inspection and downstream integration. |

The notebook uses cached outputs for bundled recording samples where possible, which protects the submission guardrail: evaluators can run the demo even when GPU, model downloads, `ffmpeg`, or browser microphone permissions are unreliable.

Demo assets are grouped by purpose under `data/demo/`: `audio/` for MP3 samples, `catalog/` for products/promotions, `captions/` for cached transcripts, and `actions/` for cached commerce actions.


### 1.2 Process Pipeline

```text
Audio source
  -> pause-aware chunking / cached sample lookup
  -> ASR caption segments: {start, end, text, source, confidence}
  -> timestamp repair and validation
  -> transcript history window for current decision
  -> product/promo candidate retrieval
  -> deterministic decision provider
  -> catalog and promotion validation
  -> timestamped commerce actions
  -> synchronized viewer UI cards and JSON outputs
```

**Recording mode:** processes the full audio first, then renders captions and actions against playback time. Audio 1 and Audio 2 use bundled cached captions/actions for deterministic evaluation.

**Live file mode:** treats an audio file as a live stream. Chunks are released only when the simulated playback time reaches that chunk, so ASR/action inference cannot run ahead of the stream.

**Live mic mode:** continuously records browser microphone audio into a queue while ASR and action extraction consume chunks independently. Silent chunks are skipped to reduce random hallucinated captions.


### 1.3 Models and Core AI Logic

**ASR model.** The default demo ASR is `openai_whisper` with model `turbo`, language `th`, chunk length up to 15 seconds, pause threshold 0.3 seconds, and silence threshold 0.015. Recording mode can use cached sample outputs; uploaded audio and mic input can run real ASR through the OpenAI Whisper package.

**Chunking logic.** Dynamic chunking searches for speaker pauses instead of blindly cutting fixed windows. This improves live behavior when product, bundle, or promo wording spans uneven speech. If no suitable pause is found before the max chunk length, the system still emits a bounded chunk so latency stays controlled.

**Action-decision logic.** The action engine keeps a rolling transcript history. If no action is emitted, recent prior segments remain available for the next decision so split phrases such as ?use cleanser first? and ?then serum? can become one bundle decision. Once an action is emitted, that history resets to avoid repeatedly firing on old evidence. If history grows beyond a fixed character budget, oldest text is dropped first.

**Retrieval logic.** Products are searched by SKU, product name, brand, category, description, tags, and compatible SKUs. Promotions are searched by code, description, discount type/value, live-only flag, eligible categories/tags/SKUs. The retriever normalizes Thai digits and tone marks, scores character n-grams and substrings, and boosts candidates when spoken prices or promo numbers match catalog data.

**Validation logic.** AI can suggest actions, but catalog rules decide what is safe to emit. Unknown SKUs, unsupported promo codes, promo-ineligible products, and incompatible bundles are rejected before becoming UI actions.

**Action decision.** `src/ai/decision.py` uses a deterministic provider for the demo. It receives the current transcript window, unresolved previous text, active session state, product candidates, and promotion candidates, then emits at most one structured decision per caption segment. `src/ai/commerce_actions.py` validates each decision before rendering product cards, promo cards, bundle cards, or flash-sale countdowns.


In [ ]:
import contextlib
import io
import os
import sys
import subprocess
from pathlib import Path

from IPython.display import HTML, display

REPO_URL = "https://github.com/Siratish/Live-Commerce-Copilot.git"
REPO_DIR_NAME = "Live-Commerce-Copilot"


def setup_card(title, detail=""):
    display(HTML(
        '<div style="border:1px solid #bfdbfe;background:#eff6ff;color:#1e3a8a;'
        'border-radius:8px;padding:12px;font-family:Arial,sans-serif;margin-bottom:10px;">'
        f'<strong>{title}</strong>'
        f'<div style="font-size:13px;line-height:1.45;margin-top:4px;">{detail}</div>'
        '</div>'
    ))


def looks_like_project(path: Path) -> bool:
    return (path / "config" / "demo.yaml").exists() and (path / "src").exists()


repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if looks_like_project(candidate):
        repo_root = candidate
        break

if repo_root is None:
    clone_parent = Path("/content") if Path("/content").exists() else Path.cwd()
    repo_root = clone_parent / REPO_DIR_NAME
    if repo_root.exists() and not looks_like_project(repo_root):
        raise RuntimeError(f"{repo_root} exists but does not look like the target project.")
    if not repo_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo_root)], stdout=subprocess.DEVNULL)

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

requirements = repo_root / "requirements.txt"
if requirements.exists():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)]
    )

try:
    import ipywidgets  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])

from src.utils.live_mic import preload_openai_whisper_model

with contextlib.redirect_stdout(io.StringIO()):
    PRELOADED_OPENAI_WHISPER_TURBO = preload_openai_whisper_model("turbo")

setup_card(
    "Notebook setup complete",
    f"Repo ready at <code>{repo_root}</code>. OpenAI Whisper turbo is loaded for the demo.",
)


## 2. Catalog UI

The catalog is the commerce source of truth. Products and promotions are stored separately so promo codes can be governed by eligibility rules such as category, tag, SKU, live-only status, and stock requirements. The UI below lets evaluators inspect the demo data and add temporary notebook-session products or promotions.

The product schema includes SKU, name, brand, category, original price, live price, stock, tags, compatible SKUs, description, and deeplink. The promotion schema includes code, description, discount type/value, live-only flag, eligible categories/tags/SKUs, and minimum-stock requirement. This structure avoids attaching one promo code permanently to each product and makes campaign rules easier to scale.

In production, this layer would connect to True commerce/catalog systems, campaign tools, seller portals, and inventory services. The action engine can suggest products or promos, but validation must happen against the catalog and promotion rules before anything is shown to viewers.


In [ ]:
# CATALOG_MANAGER_CELL_V1
from pathlib import Path
from src.utils.demo_notebook_ui import display_catalog_manager_ui

display_catalog_manager_ui(Path.cwd())


## 3. Recording Mode Demo

Recording mode is the safest end-to-end path for evaluation. Audio 1 and Audio 2 use cached captions/actions so the notebook always completes, while upload and microphone recordings can run through the ASR pipeline. The UI shows the mock live scene, captions, current action, and action history after processing completes.

This mode demonstrates the batch path: load audio, transcribe or load cached captions, validate monotonic timestamps, run commerce-action extraction across the transcript, write JSON outputs, and render synchronized cards during playback.

Use this mode to evaluate the business flow: spoken sales pitch -> Thai captions -> validated product/promo/bundle/countdown actions -> viewer-facing commerce cards.


In [ ]:
from pathlib import Path
from src.utils.demo_notebook_ui import display_recording_mode_demo_ui

display_recording_mode_demo_ui(Path.cwd())


## 4. Live Mode Demo

Live mode demonstrates the production direction: audio arrives over time, pause-aware chunks are released to ASR only after the stream reaches that point, silent chunks are skipped, and commerce actions are emitted as the transcript accumulates enough evidence.

The live implementation uses a producer/consumer style queue. Audio capture or playback continues independently from ASR/action processing, so a slow model does not pause input capture. If chunks arrive faster than inference can consume them, the queue buffers work until it drains.

A production deployment would split this into services: stream ingestion, ASR workers, action-decision workers, catalog/promotion validation, and a realtime API or event bus for the viewer UI. CI/CD would run unit tests and notebook smoke checks; model workers could autoscale on GPU capacity; observability would track latency, queue depth, ASR quality, action precision, and conversion events.


### 4.1 Audio Source: File

This live demo uses Audio 1, Audio 2, or an uploaded audio file as a simulated live stream. The system cannot process a chunk before playback time passes that chunk, which mirrors how a real stream would behave while still being repeatable in Colab.


In [ ]:
from pathlib import Path
from src.utils.demo_notebook_ui import display_live_audio_file_demo_ui

display_live_audio_file_demo_ui(Path.cwd())


### 4.2 Audio Source: Mic

This live demo streams browser microphone input into the same ASR and action pipeline. Use it to test ad-libbed Thai or English selling speech, product mentions, promo cues, bundle intent, and flash-sale urgency in a more realistic host workflow.


In [ ]:
from pathlib import Path
from src.utils.demo_notebook_ui import display_live_mic_demo_ui

display_live_mic_demo_ui(Path.cwd())
